In [1]:
!pip install numpy matplotlib ipywidgets --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 20.7 MB/s eta 0:00:00


## Quantum Walk vs Classical Walk

This script provides an interactive slider where your students can pick the number of steps ($N$). When they click "Run Quantum Walk", it animates the walker on a number line step-by-step, showing where the quantum state is at each moment. Once the animation finishes, it plots the final probability distribution graph showing the peak at the outer edges!

In [4]:
import time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

def run_quantum_walk_simulation(num_steps):
    """
    Simulates a 1D Discrete Quantum Walk for N steps with auto-scaling UI to prevent label overlap.
    """
    positions = np.arange(-num_steps, num_steps + 1)
    num_positions = len(positions)
    zero_idx = num_steps

    state = np.zeros((2, num_positions), dtype=complex)
    state[0, zero_idx] = 1.0 + 0j

    H = (1 / np.sqrt(2)) * np.array([[1, 1],
                                     [1, -1]])

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8.5), gridspec_kw={'height_ratios': [1.2, 2]})
    plt.ion()

    # --- ANIMATION LOOP ---
    for t in range(num_steps + 1):
        prob = np.sum(np.abs(state)**2, axis=0)

        ax1.clear()
        ax2.clear()

        # ----------------------------------------------------
        # TOP PANEL: Number Line (Cleaned Labels & Rotated Text)
        # ----------------------------------------------------
        ax1.set_xlim(-num_steps - 1.5, num_steps + 1.5)
        ax1.set_ylim(-0.5, 2.2) # Increased Y-limit to give room for text
        ax1.axhline(0, color='black', linewidth=1.5)

        # Filter active positions
        active_indices = np.where(prob > 0.001)[0]
        active_positions = positions[active_indices]
        active_probs = prob[active_indices]

        # Plot pointers/dots on the number line
        ax1.scatter(active_positions, np.zeros_like(active_positions),
                    s=active_probs * 600 + 40, color='red', zorder=3)

        # Annotate percentages (Rotated 90 degrees & staggered heights to avoid overlap)
        for i, (pos, p) in enumerate(zip(active_positions, active_probs)):
            y_offset = 0.35 if (i % 2 == 0) else 0.85 # Alternating heights
            ax1.annotate(f"{p*100:.1f}%", (pos, y_offset), ha='center', va='bottom',
                         fontsize=8 if num_steps > 10 else 9,
                         fontweight='bold', color='darkred', rotation=90)

        # Clean x-ticks spacing dynamically
        tick_step = 1 if num_steps <= 8 else (2 if num_steps <= 15 else 5)
        tick_locations = np.arange(-num_steps, num_steps + 1, tick_step)
        ax1.set_xticks(tick_locations)
        ax1.set_yticks([])
        ax1.set_title(f"Step {t} / {num_steps}: Walker Position on Number Line", fontsize=12, fontweight='bold')
        ax1.grid(True, axis='x', linestyle='--', alpha=0.4)

        # ----------------------------------------------------
        # BOTTOM PANEL: Probability Graph
        # ----------------------------------------------------
        ax2.bar(positions, prob, color='indigo', alpha=0.75, edgecolor='black', width=0.7)
        ax2.set_xlim(-num_steps - 1.5, num_steps + 1.5)

        # Dynamic Y-limit scaling based on maximum current probability
        max_p = np.max(prob)
        ax2.set_ylim(0, max(0.3, max_p * 1.2))

        ax2.set_xticks(tick_locations)
        ax2.set_xlabel("Position on Line (x)", fontsize=11)
        ax2.set_ylabel("Probability P(x)", fontsize=11)
        ax2.set_title("Quantum Probability Distribution Graph", fontsize=12)
        ax2.grid(True, linestyle=':', alpha=0.6)

        plt.tight_layout()
        clear_output(wait=True)
        display(fig)
        plt.close(fig)

        time.sleep(0.6 if num_steps <= 5 else 0.25)

        # --- EVOLVE STATE TO NEXT STEP ---
        if t < num_steps:
            new_state = np.zeros_like(state)
            for pos_i in range(num_positions):
                new_state[:, pos_i] = H @ state[:, pos_i]

            shifted_state = np.zeros_like(new_state)
            for pos_i in range(num_positions):
                if pos_i > 0:
                    shifted_state[0, pos_i - 1] += new_state[0, pos_i]
                if pos_i < num_positions - 1:
                    shifted_state[1, pos_i + 1] += new_state[1, pos_i]

            state = shifted_state

# --- INTERACTIVE WIDGET UI SETUP ---
steps_slider = widgets.IntSlider(
    value=20,
    min=1,
    max=30,
    step=1,
    description='Steps (N):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

run_button = widgets.Button(
    description='Run Quantum Walk',
    button_style='success',
    icon='play'
)

output_area = widgets.Output()

def on_button_clicked(b):
    with output_area:
        clear_output()
        run_quantum_walk_simulation(steps_slider.value)

run_button.on_click(on_button_clicked)

display(widgets.VBox([
    widgets.HTML("<h3>Interactive Quantum Walk Visualizer (Clean Layout)</h3>"),
    widgets.HBox([steps_slider, run_button]),
    output_area
]))

<Figure size 640x480 with 0 Axes>